In [88]:
import numpy as np

In [89]:
# Clase neurona
class NetNode(object):
  # Esta clase representa un perceptrón
  def __init__(self):
    self.inputs = []
    self.weights = []
    self.value = None

In [90]:
class Network(object):
    # Clase red neuronal, que permite construirla y entrenarla
    def __init__(self, layers):
        """
        layers: es una lista que representa las neuronas y capas de la red.
        Por ejemplo, [2,3,1]
        Significa que hay 3 capas, una por cada elemento de la lista:
        - input layer: 2 neuronas.
        - hidden layer: 3 neuronas.
        - output layer: 1 neurona.
        """
        self.net = [[NetNode() for _ in range(size)] for size in layers]

        sizes = len(layers)

        # Creo las conexiones
        for layer in range(1, sizes):
            for node in self.net[layer]:
                for unit in self.net[layer-1]:
                    node.inputs.append(unit)
                    node.weights.append(0)

    def relu(self, z):
        return max(0, z)

    def relu_prime(self, z):
        return 1 if z > 0 else 0

    def predict(self, input_data):
        inputs = self.net[0]

        # Initialize inputs
        for v, n in zip(input_data, inputs):
            n.value = v

        # Forward step
        for layer in self.net[1:]:
            for node in layer:
                in_val = [n.value for n in node.inputs]
                unit_value = np.dot(in_val, node.weights)
                node.value = self.relu(unit_value)

        outputs = self.net[-1]
        return outputs.index(max(outputs, key=lambda node: node.value))

    def accuracy(self, examples):
        correct = 0

        for x_test, y_test in examples:
            prediction = self.predict(x_test)

            if (y_test[prediction] == 1):
                correct += 1

        return correct / len(examples)

    def backpropagation(self, eta, examples, epochs):
        inputs = self.net[0]
        outputs = self.net[-1]
        layer_size = len(self.net)

        # Initialize weights
        for layer in self.net[1:]:
            for node in layer:
                node.weights = [np.random.uniform() for _ in range(len(node.weights))]

        for epoch in range(epochs):
            # Variable para acumular el error de la época (opcional para el print final)
            epoch_err = 0

            for x_train, y_train in examples:
                # Initialize inputs
                for value, node in zip(x_train, inputs):
                    node.value = value

                # Forward step
                for layer in self.net[1:]:
                    for node in layer:
                        in_val = [n.value for n in node.inputs]
                        unit_value = np.dot(in_val, node.weights)
                        node.value = self.relu(unit_value)

                # Initialize delta
                delta = [[] for _ in range(layer_size)]

                # Error for the MSE cost function
                err = [y_train[i] - outputs[i].value for i in range(len(outputs))]

                # Acumulamos error para visualización
                epoch_err += np.sum([e**2 for e in err])

                delta[-1] = [self.relu_prime(outputs[i].value) * err[i] for i in range(len(outputs))]

                # Backward step
                hidden_layers = layer_size - 2
                for i in range(hidden_layers, 0, -1):
                    layer = self.net[i]
                    n_layers = len(layer)

                    # Weights from the last layer
                    w = [[node.weights[l] for node in self.net[i + 1]] for l in range(n_layers)]

                    delta[i] = [self.relu_prime(layer[j].value) * np.dot(w[j], delta[i + 1]) for j in range(n_layers)]

                # Update weights
                for i in range(1, layer_size):
                    layer = self.net[i]
                    in_val = [node.value for node in self.net[i - 1]]
                    n_layers = len(self.net[i])
                    for j in range(n_layers):
                        layer[j].weights = np.add(layer[j].weights, np.multiply(eta * delta[i][j], in_val))

            print(f"epoch {epoch}/{epochs} | total error={epoch_err/len(examples)}")

In [91]:
pip install keras

In [92]:
from sklearn import datasets
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

In [93]:
# importamos los datos
iris_X, iris_y = datasets.load_iris(return_X_y=True)

In [94]:
# Mostrar los 10 primeros elementos
iris_X[:10]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1]])

In [95]:
# Mostrar las 10 primeras clases(output)
iris_y[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [96]:
iris_X_normalized = normalize(iris_X, axis=0)

In [97]:
# Mostrar los 10 primeros elementos normalizados
iris_X_normalized[:10]

array([[0.07056264, 0.09254209, 0.02754801, 0.01150242],
       [0.06779548, 0.07932179, 0.02754801, 0.01150242],
       [0.06502832, 0.08460991, 0.02558029, 0.01150242],
       [0.06364474, 0.08196585, 0.02951572, 0.01150242],
       [0.06917906, 0.09518615, 0.02754801, 0.01150242],
       [0.07471338, 0.10311833, 0.03345115, 0.02300485],
       [0.06364474, 0.08989803, 0.02754801, 0.01725364],
       [0.06917906, 0.08989803, 0.02951572, 0.01150242],
       [0.06087757, 0.07667773, 0.02754801, 0.01150242],
       [0.06779548, 0.08196585, 0.02951572, 0.00575121]])

In [98]:
# Dividir en entrenamiento y test
# 80 - 20

X_train, X_test, y_train, y_test = train_test_split(
    iris_X_normalized, iris_y, test_size=0.2, shuffle=True
)

In [99]:
# Convertir las clases de categorías ('Setosa', 'Versicolor', 'Virginica')
# a númericas (0, 1, 2) y luego hacer one-hot encoded
# ([1,0,0], [0,1,0], [0,0,1])

y_train = to_categorical(y_train, num_classes=3)
y_test = to_categorical(y_test, num_classes=3)

In [100]:
y_train [:10]


array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.]])

In [101]:
examples = []
for i in range(len(X_train)):
    examples.append([X_train[i], y_train[i]])

    print(examples)

[[array([0.07609697, 0.06874555, 0.08657946, 0.06901454]), array([0., 1., 0.])]]
[[array([0.07609697, 0.06874555, 0.08657946, 0.06901454]), array([0., 1., 0.])], [array([0.10653575, 0.10047427, 0.1318369 , 0.12652666]), array([0., 0., 1.])]]
[[array([0.07609697, 0.06874555, 0.08657946, 0.06901454]), array([0., 1., 0.])], [array([0.10653575, 0.10047427, 0.1318369 , 0.12652666]), array([0., 0., 1.])], [array([0.07609697, 0.09254209, 0.02558029, 0.01150242]), array([1., 0., 0.])]]
[[array([0.07609697, 0.06874555, 0.08657946, 0.06901454]), array([0., 1., 0.])], [array([0.10653575, 0.10047427, 0.1318369 , 0.12652666]), array([0., 0., 1.])], [array([0.07609697, 0.09254209, 0.02558029, 0.01150242]), array([1., 0., 0.])], [array([0.06917906, 0.09518615, 0.02754801, 0.01150242]), array([1., 0., 0.])]]
[[array([0.07609697, 0.06874555, 0.08657946, 0.06901454]), array([0., 1., 0.])], [array([0.10653575, 0.10047427, 0.1318369 , 0.12652666]), array([0., 0., 1.])], [array([0.07609697, 0.09254209, 0.0

In [102]:
network = Network([4, 7, 3])

In [103]:
network.backpropagation(0.1, examples, 500)

epoch 0/500 | total error=0.6795672635917415
epoch 1/500 | total error=0.654635728993177
epoch 2/500 | total error=0.6371679721173987
epoch 3/500 | total error=0.6128189577824978
epoch 4/500 | total error=0.5793778809190246
epoch 5/500 | total error=0.5370167456369113
epoch 6/500 | total error=0.4896073553560954
epoch 7/500 | total error=0.44427826258056313
epoch 8/500 | total error=0.4073130142727141
epoch 9/500 | total error=0.38064410139250104
epoch 10/500 | total error=0.3629574224178355
epoch 11/500 | total error=0.35114054791211974
epoch 12/500 | total error=0.3428196455877038
epoch 13/500 | total error=0.3367099563612292
epoch 14/500 | total error=0.3318268384674076
epoch 15/500 | total error=0.3277412879244125
epoch 16/500 | total error=0.3242304669137364
epoch 17/500 | total error=0.3210881524251928
epoch 18/500 | total error=0.31820045572494643
epoch 19/500 | total error=0.31550067458546577
epoch 20/500 | total error=0.3129243659746083
epoch 21/500 | total error=0.31046632719

In [104]:
examples = []
for i in range(len(X_test)):
  examples.append([X_test[i], y_test[i]])

In [105]:
accuracy = network.accuracy(examples)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9666666666666667


In [106]:
prediction = network.predict(X_test[2])
print(f"Desired output: {y_test[2]}")
print(f"Index of output: {prediction}")

Desired output: [0. 1. 0.]
Index of output: 1
